# 06 — Hyperparameter Tuning + Ensemble
**Owner:** Panashe  |  **Phase:** 5-6  |  **Date:** May 14-15

Objectives: tune LightGBM with Optuna, compare simple avg / weighted avg / stacking.

In [ ]:
import sys, pathlib
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
sys.path.insert(0, str(pathlib.Path(".").resolve()))

from src.preprocessing import load_processed
from src.train import load_folds, cross_validate_model
from src.models import get_lgbm
from src.config import TARGET_COL, ID_COL, RANDOM_SEED

train_proc, test_proc = load_processed()
FEATURE_COLS = [c for c in train_proc.columns if c not in [TARGET_COL, ID_COL]]
X = train_proc[FEATURE_COLS].values
y = train_proc[TARGET_COL].values
folds = load_folds()

## 1. Optuna Tuning — LightGBM (100 trials)

In [ ]:
def lgbm_objective(trial):
    params = {
        "num_leaves":        trial.suggest_int("num_leaves", 20, 200),
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth":         trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }
    results = cross_validate_model(get_lgbm(**params), X, y, folds)
    return results["oof_auc"]

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study.optimize(lgbm_objective, n_trials=100)
print(f"Best AUC: {study.best_value:.5f}")
print(f"Best params: {study.best_params}")

## 2. Load OOF and Test Predictions

In [ ]:
lgbm_oof      = np.load("models/lgbm_oof.npy")
xgb_oof       = np.load("models/xgb_oof.npy")
catboost_oof  = np.load("models/catboost_oof.npy")
lgbm_test     = np.load("models/lgbm_test.npy")
xgb_test      = np.load("models/xgb_test.npy")
catboost_test = np.load("models/catboost_test.npy")

print("Individual OOF AUCs:")
print(f"  LightGBM:  {roc_auc_score(y, lgbm_oof):.5f}")
print(f"  XGBoost:   {roc_auc_score(y, xgb_oof):.5f}")
print(f"  CatBoost:  {roc_auc_score(y, catboost_oof):.5f}")

## 3. Simple Average

In [ ]:
avg_oof  = (lgbm_oof + xgb_oof + catboost_oof) / 3
avg_test = (lgbm_test + xgb_test + catboost_test) / 3
print(f"Simple average OOF AUC: {roc_auc_score(y, avg_oof):.5f}")

## 4. Weighted Average (by OOF AUC)

In [ ]:
w1, w2, w3 = roc_auc_score(y, lgbm_oof), roc_auc_score(y, xgb_oof), roc_auc_score(y, catboost_oof)
total = w1 + w2 + w3
wtd_oof  = (w1*lgbm_oof  + w2*xgb_oof  + w3*catboost_oof)  / total
wtd_test = (w1*lgbm_test + w2*xgb_test + w3*catboost_test) / total
print(f"Weighted average OOF AUC: {roc_auc_score(y, wtd_oof):.5f}")

## 5. Stacking (Logistic Regression meta-learner)

In [ ]:
oof_stack  = np.column_stack([lgbm_oof, xgb_oof, catboost_oof])
test_stack = np.column_stack([lgbm_test, xgb_test, catboost_test])
meta = LogisticRegression(C=1.0, random_state=RANDOM_SEED)
meta.fit(oof_stack, y)
stack_oof  = meta.predict_proba(oof_stack)[:,1]
stack_test = meta.predict_proba(test_stack)[:,1]
print(f"Stacking OOF AUC: {roc_auc_score(y, stack_oof):.5f}")

## 6. Pick Best and Save
Update `best_test_preds` with the ensemble that gave the highest OOF AUC.

In [ ]:
# best_test_preds = avg_test     # or wtd_test or stack_test
# np.save("models/best_ensemble_test.npy", best_test_preds)